# Impairment Probability of Default (PD) Model Development

This notebook demonstrates a 3-step showcase for developing an impairment PD model:
1. **Data Simulation** — Generate 10,000 synthetic loan-level observations
2. **Default Event Simulation** — Score each observation and simulate a binary default flag (5% default rate)
3. **Model Fitting** — Fit a logistic regression model to predict default

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, classification_report
from scipy.special import expit
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True

---
## Step 1: Data Simulation

Generate 10,000 synthetic observations with seven raw features, then engineer four piecewise-linear spline terms from `perf_rel_t` at knots 2, 3, and 18.

In [ ]:
np.random.seed(42)
n_samples = 10000

# ── Raw features ──────────────────────────────────────────────────────────────
df = pd.DataFrame({
    'perf_rel_t'         : np.clip(np.random.exponential(scale=25, size=n_samples), 1, 84).astype(int),
    'ecm_0106_tot_num_T' : np.clip(np.random.normal(1.068551, 0.595350, n_samples), 0, 2.079442),
    'ecm_0601_tot_num_T' : np.clip(np.random.exponential(scale=2, size=n_samples), 0, 15).astype(int),
    'fico_snp_T'         : np.clip(np.random.normal(2825, 1495, n_samples), 442, 6132),
    'util_snp_T'         : np.clip(np.random.normal(1.74, 0.65, n_samples), 1.06, 2.87),
    'num_la_snp_6mo_T'   : np.clip(np.random.poisson(2, n_samples), 0, 5),
    'UR_M_3MGR_PIT_L6_NT': np.clip(np.random.normal(-0.02, 0.038, n_samples), -0.156, 0.204),
})

# ── Piecewise-linear spline transforms of perf_rel_t ─────────────────────────
x = df['perf_rel_t']

# T1 = min(x, 2)  — capped at 2
xi = x.fillna(0, inplace=False).copy()
xi[x >= 2] = 2
df['perf_rel_t_T1'] = xi

# T2 = min(max(0, x - 2), 1)  — knot at 2, capped at 1
xi = x.fillna(0, inplace=False)
xi = xi.apply(lambda v: 0 if v <= 2 else v - 2)
xi.clip(upper=1, inplace=True)
df['perf_rel_t_T2'] = xi

# T3 = min(max(0, x - 3), 15)  — knot at 3, capped at 15
xi = x.fillna(0, inplace=False)
xi = xi.apply(lambda v: 0 if v <= 3 else v - 3)
xi.clip(upper=15, inplace=True)
df['perf_rel_t_T3'] = xi

# T4 = max(0, x - 18)  — knot at 18, uncapped
xi = x.fillna(0, inplace=False)
xi = xi.apply(lambda v: 0 if v <= 18 else v - 18)
df['perf_rel_t_T4'] = xi

print(f'Dataset shape: {df.shape}')
df.describe().round(4)

In [ ]:
# Visualise key raw-feature distributions
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
cols = ['perf_rel_t', 'ecm_0106_tot_num_T', 'ecm_0601_tot_num_T',
        'fico_snp_T', 'util_snp_T', 'num_la_snp_6mo_T', 'UR_M_3MGR_PIT_L6_NT']
for ax, col in zip(axes.flat, cols):
    ax.hist(df[col], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    ax.set_title(col, fontsize=9)
axes.flat[-1].set_visible(False)
fig.suptitle('Step 1 — Simulated Feature Distributions', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 2: Default Event Simulation

Apply the pre-estimated logistic regression coefficients to produce a linear predictor (log-odds score) for each observation.  
A small amount of Gaussian noise is added to introduce realistic randomness.  
The top-scoring **5%** of observations are labelled `default_event = 1`.

In [ ]:
# ── Pre-estimated model coefficients ─────────────────────────────────────────
coef = {
    'const'              : -7.1430,
    'perf_rel_t_T1'      :  1.9057,
    'perf_rel_t_T2'      :  0.0157,
    'perf_rel_t_T3'      : -0.0303,
    'perf_rel_t_T4'      : -0.0054,
    'ecm_0106_tot_num_T' :  0.2352,
    'ecm_0601_tot_num_T' :  0.0329,
    'fico_snp_T'         : -0.0004,
    'util_snp_T'         :  0.4751,
    'num_la_snp_6mo_T'   :  0.1048,
    'UR_M_3MGR_PIT_L6_NT':  3.3694,
}

# ── Compute linear predictor ─────────────────────────────────────────────────
score = coef['const']
for feat, weight in coef.items():
    if feat != 'const':
        score = score + weight * df[feat]

df['logit_score'] = score
df['pd_prob']     = expit(score)   # sigmoid → estimated PD

print(f"Score range  : [{score.min():.3f}, {score.max():.3f}]")
print(f"Raw PD range : [{df['pd_prob'].min():.4f}, {df['pd_prob'].max():.4f}]")
print(f"Mean raw PD  : {df['pd_prob'].mean():.4f}")

In [ ]:
# ── Simulate default_event with ~5% default rate ──────────────────────────────
# Add Gaussian noise to the score so assignment is probabilistic, not deterministic
np.random.seed(42)
noise = np.random.normal(0, score.std() * 0.5, n_samples)
noisy_score = score + noise

n_defaults = int(round(0.05 * n_samples))          # exactly 500 defaults
default_idx = np.argsort(noisy_score)[-n_defaults:] # top-scoring rows → default

df['default_event'] = 0
df.loc[default_idx, 'default_event'] = 1

print(f"Total defaults : {df['default_event'].sum()}")
print(f"Default rate   : {df['default_event'].mean():.2%}")

In [ ]:
# Score distribution by default status
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for flag, label, color in [(0, 'Non-default', 'steelblue'), (1, 'Default', 'tomato')]:
    axes[0].hist(df.loc[df['default_event'] == flag, 'logit_score'],
                 bins=50, alpha=0.7, label=label, color=color, density=True)
axes[0].set_title('Logit Score Distribution by Default Status')
axes[0].set_xlabel('Logit Score')
axes[0].legend()

for flag, label, color in [(0, 'Non-default', 'steelblue'), (1, 'Default', 'tomato')]:
    axes[1].hist(df.loc[df['default_event'] == flag, 'pd_prob'],
                 bins=50, alpha=0.7, label=label, color=color, density=True)
axes[1].set_title('PD Probability Distribution by Default Status')
axes[1].set_xlabel('Estimated PD (sigmoid of score)')
axes[1].legend()

fig.suptitle('Step 2 — Simulated Default Event', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── PD Probability Histogram by Default Event ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

for flag, label, color in [(0, 'Non-default (0)', 'steelblue'), (1, 'Default (1)', 'tomato')]:
    ax.hist(
        df.loc[df['default_event'] == flag, 'pd_prob'],
        bins=60, alpha=0.7, label=label, color=color, density=True, edgecolor='white'
    )

ax.set_title('PD Probability Distribution by Default Event', fontsize=13, fontweight='bold')
ax.set_xlabel('pd_prob (Estimated PD)')
ax.set_ylabel('Density')
ax.legend(title='default_event')
plt.tight_layout()
plt.show()


---
## Step 3: Logistic Regression Model Fitting

Fit a logistic regression model using all 11 features created in Step 1 as predictors and `default_event` as the target.  
We use **statsmodels** for the full coefficient summary and **scikit-learn** for AUC-ROC evaluation.

In [ ]:
# ── Variance Inflation Factor (VIF) for each feature ─────────────────────────
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = sm.add_constant(X)
vif_df = pd.DataFrame({
    'Feature': X_vif.columns,
    'VIF'    : [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])],
}).set_index('Feature').drop('const')

vif_df['VIF'] = vif_df['VIF'].round(2)
vif_df = vif_df.sort_values('VIF', ascending=False)

# Colour-coded bar chart
colors = ['tomato' if v >= 10 else 'orange' if v >= 5 else 'steelblue' for v in vif_df['VIF']]
fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(vif_df.index, vif_df['VIF'], color=colors, edgecolor='white')
ax.axvline(5,  color='orange', linestyle='--', lw=1.2, label='VIF = 5 (moderate)')
ax.axvline(10, color='tomato',  linestyle='--', lw=1.2, label='VIF = 10 (high)')
ax.bar_label(bars, fmt='%.1f', padding=3, fontsize=8)
ax.set_xlabel('VIF')
ax.set_title('Variance Inflation Factor by Feature', fontsize=13, fontweight='bold')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print('\nVIF Table:')
print(vif_df.to_string())


In [ ]:
# ── Define predictors (all columns created in Step 1) ────────────────────────
feature_cols = [
    'perf_rel_t',
    'ecm_0106_tot_num_T',
    'ecm_0601_tot_num_T',
    'fico_snp_T',
    'util_snp_T',
    'num_la_snp_6mo_T',
    'UR_M_3MGR_PIT_L6_NT',
    'perf_rel_t_T1',
    'perf_rel_t_T2',
    'perf_rel_t_T3',
    'perf_rel_t_T4',
]

X = df[feature_cols]
y = df['default_event']

In [ ]:
# ── statsmodels: full coefficient table with p-values ────────────────────────
X_sm = sm.add_constant(X)
sm_model = sm.Logit(y, X_sm)
sm_result = sm_model.fit(maxiter=200, disp=True)
print(sm_result.summary())

In [ ]:
# ── Coefficient comparison: fitted vs. pre-estimated ─────────────────────────
fitted_coefs = sm_result.params

# Map pre-estimated coefs to the same index names
pre_est = {
    'const'              : -7.1430,
    'perf_rel_t_T1'      :  1.9057,
    'perf_rel_t_T2'      :  0.0157,
    'perf_rel_t_T3'      : -0.0303,
    'perf_rel_t_T4'      : -0.0054,
    'ecm_0106_tot_num_T' :  0.2352,
    'ecm_0601_tot_num_T' :  0.0329,
    'fico_snp_T'         : -0.0004,
    'util_snp_T'         :  0.4751,
    'num_la_snp_6mo_T'   :  0.1048,
    'UR_M_3MGR_PIT_L6_NT':  3.3694,
}

compare_df = pd.DataFrame({
    'Fitted'       : fitted_coefs,
    'Pre-estimated': pd.Series(pre_est, name='Pre-estimated'),
}).dropna()

compare_df['Difference'] = (compare_df['Fitted'] - compare_df['Pre-estimated']).round(4)
print('\nCoefficient Comparison')
print(compare_df.round(4).to_string())

In [ ]:
# ── Lift Chart: pd_prob discriminative power vs default_event ─────────────────
n_buckets = 10

lift_df = df[['pd_prob', 'default_event']].copy()
lift_df = lift_df.sort_values('pd_prob', ascending=False).reset_index(drop=True)

# Assign decile buckets (1 = highest PD score, 10 = lowest)
lift_df['decile'] = pd.qcut(lift_df.index, q=n_buckets, labels=range(1, n_buckets + 1))

baseline_rate = lift_df['default_event'].mean()

bucket_stats = (
    lift_df.groupby('decile', observed=True)['default_event']
    .agg(total='count', defaults='sum')
    .assign(
        default_rate=lambda d: d['defaults'] / d['total'],
        lift=lambda d: (d['defaults'] / d['total']) / baseline_rate,
        cum_defaults=lambda d: d['defaults'].cumsum(),
        cum_pct_population=lambda d: d['total'].cumsum() / lift_df.shape[0],
        cum_capture_rate=lambda d: d['defaults'].cumsum() / d['defaults'].sum(),
    )
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Panel 1: Lift by decile ───────────────────────────────────────────────────
colors = ['tomato' if v >= 2 else 'steelblue' for v in bucket_stats['lift']]
axes[0].bar(bucket_stats.index.astype(str), bucket_stats['lift'], color=colors, edgecolor='white')
axes[0].axhline(1.0, color='black', linestyle='--', lw=1.2, label='Baseline (random)')
axes[0].set_xlabel('Decile (1 = Highest PD Score)')
axes[0].set_ylabel('Lift')
axes[0].set_title('Lift by Score Decile')
axes[0].legend(fontsize=8)
for i, (dec, row) in enumerate(bucket_stats.iterrows()):
    axes[0].text(i, row['lift'] + 0.05, f"{row['lift']:.2f}x", ha='center', fontsize=8)

# ── Panel 2: Default capture rate (cumulative) ────────────────────────────────
axes[1].plot(bucket_stats['cum_pct_population'] * 100,
             bucket_stats['cum_capture_rate'] * 100,
             marker='o', color='steelblue', lw=2, label='Model')
axes[1].plot([0, 100], [0, 100], color='grey', linestyle='--', lw=1.2, label='Random')
axes[1].set_xlabel('% Population (by descending PD score)')
axes[1].set_ylabel('% Defaults Captured')
axes[1].set_title('Cumulative Capture Rate (Lorenz Curve)')
axes[1].legend(fontsize=8)

# ── Panel 3: Default rate per decile ─────────────────────────────────────────
axes[2].bar(bucket_stats.index.astype(str), bucket_stats['default_rate'] * 100,
            color='steelblue', edgecolor='white')
axes[2].axhline(baseline_rate * 100, color='tomato', linestyle='--', lw=1.2,
                label=f'Baseline ({baseline_rate:.1%})')
axes[2].set_xlabel('Decile (1 = Highest PD Score)')
axes[2].set_ylabel('Default Rate (%)')
axes[2].set_title('Actual Default Rate by Score Decile')
axes[2].legend(fontsize=8)
for i, (dec, row) in enumerate(bucket_stats.iterrows()):
    axes[2].text(i, row['default_rate'] * 100 + 0.3,
                 f"{row['default_rate']:.1%}", ha='center', fontsize=7.5)

fig.suptitle('Lift Analysis — pd_prob vs default_event', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nLift Table by Decile:')
print(bucket_stats[['total', 'defaults', 'default_rate', 'lift', 'cum_capture_rate']].round(4).to_string())


In [ ]:
# ── Accuracy Plot: Average pd_prob vs Actual Default Rate by Decile ───────────
acc_df = df[['pd_prob', 'default_event']].copy()
acc_df['decile'] = pd.qcut(acc_df['pd_prob'], q=10, labels=range(1, 11))

decile_stats = (
    acc_df.groupby('decile', observed=True)
    .agg(
        avg_pd_prob=('pd_prob', 'mean'),
        actual_default_rate=('default_event', 'mean'),
        count=('default_event', 'count'),
    )
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = decile_stats['decile'].astype(int)
width = 0.35

# ── Panel 1: Grouped bar chart ────────────────────────────────────────────────
bars1 = axes[0].bar(x - width / 2, decile_stats['avg_pd_prob'] * 100,
                    width=width, label='Avg pd_prob (%)', color='steelblue', edgecolor='white')
bars2 = axes[0].bar(x + width / 2, decile_stats['actual_default_rate'] * 100,
                    width=width, label='Actual Default Rate (%)', color='tomato', edgecolor='white')

axes[0].set_xticks(x)
axes[0].set_xticklabels([f'D{i}' for i in x])
axes[0].set_xlabel('Decile (1 = Lowest PD, 10 = Highest PD)')
axes[0].set_ylabel('Rate (%)')
axes[0].set_title('Avg Predicted PD vs Actual Default Rate by Decile')
axes[0].legend()
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                 f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=7)
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                 f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=7)

# ── Panel 2: Scatter / calibration line ───────────────────────────────────────
axes[1].scatter(decile_stats['avg_pd_prob'] * 100,
                decile_stats['actual_default_rate'] * 100,
                color='steelblue', s=80, zorder=3)
for _, row in decile_stats.iterrows():
    axes[1].annotate(f"D{int(row['decile'])}",
                     (row['avg_pd_prob'] * 100, row['actual_default_rate'] * 100),
                     textcoords='offset points', xytext=(6, 3), fontsize=8)

# Perfect calibration line
lim_max = max(decile_stats['avg_pd_prob'].max(), decile_stats['actual_default_rate'].max()) * 100 * 1.1
axes[1].plot([0, lim_max], [0, lim_max], color='tomato', linestyle='--', lw=1.2, label='Perfect calibration')
axes[1].set_xlabel('Avg Predicted PD (%)')
axes[1].set_ylabel('Actual Default Rate (%)')
axes[1].set_title('Calibration: Predicted PD vs Actual Default Rate')
axes[1].legend(fontsize=8)

fig.suptitle('Accuracy Plot — pd_prob vs default_event by Decile', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nDecile Accuracy Table:')
decile_stats['diff'] = (decile_stats['avg_pd_prob'] - decile_stats['actual_default_rate']).round(4)
print(decile_stats[['decile', 'count', 'avg_pd_prob', 'actual_default_rate', 'diff']].round(4).to_string(index=False))


In [ ]:
# ── sklearn: AUC-ROC and ROC curve ───────────────────────────────────────────
sk_model = LogisticRegression(max_iter=500, random_state=42)
sk_model.fit(X, y)

y_pred_proba = sk_model.predict_proba(X)[:, 1]
auc = roc_auc_score(y, y_pred_proba)

fpr, tpr, thresholds = roc_curve(y, y_pred_proba)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC curve
axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc:.4f}')
axes[0].plot([0, 1], [0, 1], color='grey', linestyle='--', lw=1)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

# Predicted probability distribution by actual default
for flag, label, color in [(0, 'Non-default', 'steelblue'), (1, 'Default', 'tomato')]:
    axes[1].hist(y_pred_proba[y == flag], bins=50, alpha=0.7,
                 label=label, color=color, density=True)
axes[1].set_title('Predicted PD Distribution by Actual Default')
axes[1].set_xlabel('Predicted Probability')
axes[1].legend()

fig.suptitle('Step 3 — Fitted Logistic Regression Model', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nAUC-ROC : {auc:.4f}')
print(f'\nClassification Report (threshold = 0.5):')
print(classification_report(y, sk_model.predict(X), target_names=['Non-default', 'Default']))

In [ ]:
# ── Coefficient bar chart ────────────────────────────────────────────────────
coef_series = pd.Series(
    sk_model.coef_[0], index=feature_cols
).sort_values()

colors = ['tomato' if v > 0 else 'steelblue' for v in coef_series]
coef_series.plot(kind='barh', color=colors, figsize=(9, 5))
plt.axvline(0, color='black', lw=0.8)
plt.title('Fitted Logistic Regression Coefficients (sklearn)', fontweight='bold')
plt.xlabel('Coefficient value')
plt.tight_layout()
plt.show()